#**CODE ALPHA**

#**INTERNSHIP 2026**

#**TASK(02)**

#**CHAT_BOT**

**📦 CELL 1: Install Required Libraries**

In [2]:
!pip install flask flask-ngrok nltk scikit-learn pyngrok flask-cors

This downloads all the tools we need:

flask → creates our backend web server

nltk → cleans and processes text

scikit-learn → does the math for comparing questions (cosine similarity)

pyngrok → makes our Colab-hosted website accessible via a live link

flask-cors → lets our frontend and backend talk to each other safely

**💾 CELL 2: Create the Database**

In [3]:
import sqlite3

# Connect to (or create) the database file
conn = sqlite3.connect('chatbot.db', check_same_thread=False)
cursor = conn.cursor()

# Table to store FAQ questions and answers
cursor.execute('''
CREATE TABLE IF NOT EXISTS faqs (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    question TEXT NOT NULL,
    answer TEXT NOT NULL
)
''')

# Table to store every conversation (chat history)
cursor.execute('''
CREATE TABLE IF NOT EXISTS chat_history (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_question TEXT,
    bot_answer TEXT,
    match_score REAL,
    timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
)
''')

conn.commit()
print("✅ Database and tables created successfully!")

✅ Database and tables created successfully!


**📋 CELL 3: Insert Your 50 FAQs**

In [4]:
faqs_data = [
    ("what are your working hours", "We are open Monday to Saturday, 6 AM to 10 PM."),
    ("what is the membership fee", "Our monthly membership starts at $30, with discounts for annual plans."),
    ("can i cancel my membership anytime", "Yes, you can cancel anytime with a 30-day notice."),
    ("is there a free trial available", "Yes, we offer a 3-day free trial for new members."),
    ("do you offer student discounts", "Yes, students get a 15% discount with a valid student ID."),
    ("what payment methods do you accept", "We accept credit cards, debit cards, and bank transfers."),
    ("can i freeze my membership temporarily", "Yes, memberships can be paused for up to 2 months per year."),
    ("do you offer family or couple plans", "Yes, we offer discounted family and couple membership packages."),
    ("is there a registration fee", "A one-time registration fee of $10 applies to new members."),
    ("can i upgrade my membership plan later", "Yes, you can upgrade your plan anytime from your account settings."),
    ("how can i book an appointment", "You can book an appointment through our website or by calling our front desk."),
    ("do you accept walk-ins", "Walk-ins are welcome, but appointments are recommended to avoid waiting."),
    ("how do i reschedule my appointment", "You can reschedule through our app or by calling us at least 2 hours in advance."),
    ("what happens if i miss my appointment", "A missed appointment without notice may incur a small fee."),
    ("can i book multiple sessions in advance", "Yes, you can book up to 4 weeks of sessions in advance."),
    ("do you offer personal training", "Yes, we offer one-on-one personal training sessions with certified trainers."),
    ("do you have group fitness classes", "Yes, we offer yoga, zumba, and HIIT classes throughout the week."),
    ("can i choose my own trainer", "Yes, you can select a trainer based on availability and specialty."),
    ("are trainers certified", "All our trainers are certified in fitness training and first aid."),
    ("do you offer online training sessions", "Yes, we provide virtual training sessions for remote members."),
    ("what is the class schedule", "Class schedules are posted weekly on our website and app."),
    ("can beginners join advanced classes", "We recommend beginners start with foundational classes first."),
    ("do you offer specialized training for seniors", "Yes, we have low-impact fitness programs for senior members."),
    ("is there a limit to class size", "Group classes are limited to 15 participants for quality instruction."),
    ("do you provide diet plans", "Yes, our nutritionists offer customized diet plans for members."),
    ("do you offer physiotherapy services", "Yes, we have licensed physiotherapists available by appointment."),
    ("can i get a body composition analysis", "Yes, we offer InBody scans to track your fitness progress."),
    ("do you have a nutritionist on staff", "Yes, our certified nutritionist is available for consultations."),
    ("do you offer weight loss programs", "Yes, we have structured weight loss programs with trainer support."),
    ("is there a doctor available on site", "We have a visiting physician available twice a week for consultations."),
    ("do you offer mental health or stress management sessions", "Yes, we offer guided meditation and stress relief sessions."),
    ("can i get a fitness assessment before starting", "Yes, all new members receive a free initial fitness assessment."),
    ("what safety measures are in place", "All equipment is sanitized regularly and staff are trained in first aid."),
    ("is parking available at the clinic", "Yes, free parking is available for all members and visitors."),
    ("do you have locker rooms and showers", "Yes, we provide clean locker rooms and shower facilities."),
    ("is the gym wheelchair accessible", "Yes, our facility is fully wheelchair accessible."),
    ("do you have air conditioning", "Yes, the entire facility is climate-controlled for comfort."),
    ("what equipment do you have", "We have cardio machines, free weights, and strength training equipment."),
    ("do you provide towels", "Yes, clean towels are provided free of charge to all members."),
    ("is there a swimming pool", "Yes, we have an indoor heated swimming pool for members."),
    ("do you have cctv for security", "Yes, the facility is monitored 24/7 with CCTV cameras."),
    ("what should i wear during my visit", "Comfortable workout clothes and proper gym shoes are recommended."),
    ("can i bring a guest to the gym", "Yes, members can bring one guest per month for free."),
    ("how do i contact customer support", "You can email us at support@fitcare.com or call our helpline."),
    ("do you offer corporate memberships", "Yes, we offer discounted plans for corporate employee groups."),
    ("what age do i need to be to join", "Members must be at least 16 years old, or 12+ with parental consent."),
    ("do you have a mobile app", "Yes, our app is available on both Android and iOS for bookings and tracking."),
    ("is there wifi available at the facility", "Yes, free Wi-Fi is available throughout the facility."),
    ("can i get a refund if im not satisfied", "Refunds are available within 7 days of registration under our satisfaction policy."),
    ("do you offer merchandise like gym wear", "Yes, we sell branded gym wear and accessories at the front desk.")
]

# Clear old data first (so re-running this cell doesn't create duplicates)
cursor.execute("DELETE FROM faqs")

# Insert all 50 FAQs into the database
cursor.executemany("INSERT INTO faqs (question, answer) VALUES (?, ?)", faqs_data)
conn.commit()

print(f"✅ {len(faqs_data)} FAQs added to the database!")

✅ 50 FAQs added to the database!


**🧠 CELL 4: Text Cleaning + Matching Logic (The "Brain")**

In [5]:
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def clean_text(text):
    """Lowercase the text and remove punctuation"""
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

def get_best_answer(user_question):
    """
    Takes a user's question, compares it against all FAQs,
    and returns the best matching answer.
    """
    # Step 1: Get all FAQs fresh from the database
    cursor.execute("SELECT question, answer FROM faqs")
    rows = cursor.fetchall()
    questions = [row[0] for row in rows]
    answers = [row[1] for row in rows]

    # Step 2: Clean the FAQ questions
    cleaned_questions = [clean_text(q) for q in questions]

    # Step 3: Convert all questions into numeric vectors (TF-IDF)
    vectorizer = TfidfVectorizer()
    question_vectors = vectorizer.fit_transform(cleaned_questions)

    # Step 4: Clean and convert the user's question the same way
    cleaned_input = clean_text(user_question)
    user_vector = vectorizer.transform([cleaned_input])

    # Step 5: Compare user's question against every FAQ using cosine similarity
    similarities = cosine_similarity(user_vector, question_vectors)
    best_index = similarities.argmax()
    best_score = float(similarities[0][best_index])

    # Step 6: Decide the response based on how confident the match is
    if best_score < 0.3:
        answer = "Sorry, I don't understand that question. Could you rephrase it?"
    else:
        answer = answers[best_index]

    # Step 7: Save this conversation into chat_history table
    cursor.execute(
        "INSERT INTO chat_history (user_question, bot_answer, match_score) VALUES (?, ?, ?)",
        (user_question, answer, best_score)
    )
    conn.commit()

    return answer, best_score

HERE WHAT IS IN THIS CODE:
Get FAQs → pulls the latest questions/answers from the database (so if you add more FAQs later, it automatically uses them)

Clean text → removes capital letters and punctuation from FAQ questions, so "What's your policy?" and "whats your policy" are treated the same

TF-IDF vectors → turns each sentence into a list of numbers based on important words (this is how computers "understand" text mathematically)

Compare user's question → does the same cleaning + conversion to the user's typed question

Cosine similarity → measures how close the user's question is to each FAQ (closer to 1 = more similar, closer to 0 = unrelated)

Confidence check → if even the "best" match isn't similar enough (below 0.3), the bot admits it doesn't understand, instead of giving a wrong answer

Save to history → logs every question, answer, and how confident the match was, into your database

**🧪 CELL 5: Quick Test (before building the full frontend)**

In [6]:
import sqlite3
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Re-establish database connection and cursor
conn = sqlite3.connect('chatbot.db', check_same_thread=False)
cursor = conn.cursor()

def clean_text(text):
    """Lowercase the text and remove punctuation"""
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

def get_best_answer(user_question):
    """
    Takes a user's question, compares it against all FAQs,
    and returns the best matching answer.
    """
    # Step 1: Get all FAQs fresh from the database
    cursor.execute("SELECT question, answer FROM faqs")
    rows = cursor.fetchall()
    questions = [row[0] for row in rows]
    answers = [row[1] for row in rows]

    # Step 2: Clean the FAQ questions
    cleaned_questions = [clean_text(q) for q in questions]

    # Step 3: Convert all questions into numeric vectors (TF-IDF)
    vectorizer = TfidfVectorizer()
    question_vectors = vectorizer.fit_transform(cleaned_questions)

    # Step 4: Clean and convert the user's question the same way
    cleaned_input = clean_text(user_question)
    user_vector = vectorizer.transform([cleaned_input])

    # Step 5: Compare user's question against every FAQ using cosine similarity
    similarities = cosine_similarity(user_vector, question_vectors)
    best_index = similarities.argmax()
    best_score = float(similarities[0][best_index])

    # Step 6: Decide the response based on how confident the match is
    if best_score < 0.3:
        answer = "Sorry, I don't understand that question. Could you rephrase it?"
    else:
        answer = answers[best_index]

    # Step 7: Save this conversation into chat_history table
    cursor.execute(
        "INSERT INTO chat_history (user_question, bot_answer, match_score) VALUES (?, ?, ?)",
        (user_question, answer, best_score)
    )
    conn.commit()

    return answer, best_score

test_questions = [
    "how do I book a session",
    "can I bring a friend to workout",
    "what age is required to join",
    "asdkfjasldkfj random gibberish"
]

for q in test_questions:
    answer, score = get_best_answer(q)
    print(f"Q: {q}")
    print(f"A: {answer}  (confidence: {score:.2f})")
    print("-" * 50)


Q: how do I book a session
A: You can book an appointment through our website or by calling our front desk.  (confidence: 0.62)
--------------------------------------------------
Q: can I bring a friend to workout
A: Yes, members can bring one guest per month for free.  (confidence: 0.69)
--------------------------------------------------
Q: what age is required to join
A: Members must be at least 16 years old, or 12+ with parental consent.  (confidence: 0.75)
--------------------------------------------------
Q: asdkfjasldkfj random gibberish
A: Sorry, I don't understand that question. Could you rephrase it?  (confidence: 0.00)
--------------------------------------------------


This tests your chatbot's brain with a few example questions — including a random gibberish one on purpose, to confirm the "I don't understand" fallback works. Run this cell and check the output looks correct before moving on.

In [7]:
HTML_PAGE = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>FitCare Assistant</title>
<style>
    * { margin: 0; padding: 0; box-sizing: border-box; font-family: 'Segoe UI', sans-serif; }
    body {
        background: linear-gradient(135deg, #6366f1, #a855f7);
        height: 100vh;
        display: flex;
        align-items: center;
        justify-content: center;
    }
    .chat-container {
        width: 400px;
        height: 600px;
        background: white;
        border-radius: 20px;
        box-shadow: 0 20px 50px rgba(0,0,0,0.3);
        display: flex;
        flex-direction: column;
        overflow: hidden;
    }
    .chat-header {
        background: linear-gradient(135deg, #6366f1, #a855f7);
        color: white;
        padding: 20px;
        font-size: 1.2rem;
        font-weight: 700;
        display: flex;
        align-items: center;
        gap: 10px;
    }
    .chat-messages {
        flex: 1;
        padding: 20px;
        overflow-y: auto;
        display: flex;
        flex-direction: column;
        gap: 12px;
        background: #f8fafc;
    }
    .message {
        max-width: 75%;
        padding: 12px 16px;
        border-radius: 16px;
        font-size: 0.95rem;
        line-height: 1.4;
    }
    .bot-message {
        background: #e5e7eb;
        color: #1e293b;
        align-self: flex-start;
        border-bottom-left-radius: 4px;
    }
    .user-message {
        background: #6366f1;
        color: white;
        align-self: flex-end;
        border-bottom-right-radius: 4px;
    }
    .chat-input-area {
        display: flex;
        padding: 15px;
        border-top: 1px solid #e5e7eb;
        gap: 10px;
    }
    #userInput {
        flex: 1;
        padding: 12px 16px;
        border-radius: 25px;
        border: 1px solid #e5e7eb;
        outline: none;
        font-size: 0.95rem;
    }
    #sendBtn {
        background: #6366f1;
        color: white;
        border: none;
        border-radius: 25px;
        padding: 12px 20px;
        cursor: pointer;
        font-weight: 600;
    }
    #sendBtn:hover { background: #4f46e5; }
</style>
</head>
<body>

<div class="chat-container">
    <div class="chat-header">💪 FitCare Assistant</div>
    <div class="chat-messages" id="chatMessages">
        <div class="message bot-message">Hi! I'm FitCare Assistant 🤖 Ask me anything about memberships, classes, or trainers!</div>
    </div>
    <div class="chat-input-area">
        <input type="text" id="userInput" placeholder="Type your question...">
        <button id="sendBtn">Send</button>
    </div>
</div>

<script>
const chatMessages = document.getElementById('chatMessages');
const userInput = document.getElementById('userInput');
const sendBtn = document.getElementById('sendBtn');

function addMessage(text, sender) {
    const div = document.createElement('div');
    div.className = `message ${sender}-message`;
    div.textContent = text;
    chatMessages.appendChild(div);
    chatMessages.scrollTop = chatMessages.scrollHeight;
}

async function sendMessage() {
    const message = userInput.value.trim();
    if (!message) return;

    addMessage(message, 'user');
    userInput.value = '';

    try {
        const response = await fetch('/chat', {
            method: 'POST',
            headers: { 'Content-Type': 'application/json' },
            body: JSON.stringify({ message: message })
        });
        const data = await response.json();
        addMessage(data.reply, 'bot');
    } catch (error) {
        addMessage('Something went wrong. Please try again.', 'bot');
    }
}

sendBtn.addEventListener('click', sendMessage);
userInput.addEventListener('keypress', (e) => {
    if (e.key === 'Enter') sendMessage();
});
</script>

</body>
</html>
"""

print("✅ HTML page ready! Now run Cell 6 to launch the chatbot.")

✅ HTML page ready! Now run Cell 6 to launch the chatbot.


**🌐 CELL 7: Flask Backend (connects everything to a web page)**

In [ ]:
from flask import Flask, request, jsonify, render_template_string
from flask_cors import CORS
from pyngrok import ngrok

# HTML_PAGE is now expected to be defined in a previous cell (like Cell 7).
# If it's not defined, uncomment and use a placeholder or define it here explicitly.
# HTML_PAGE = """<!DOCTYPE html><html><head><title>Chatbot</title></head><body><h1>Hello from Chatbot!</h1></body></html>"""

app = Flask(__name__)
CORS(app)

@app.route('/')
def home():
    # This will now use the HTML_PAGE variable defined in cell mIBWUFRLAInh
    return render_template_string(HTML_PAGE)

@app.route('/chat', methods=['POST'])
def chat():
    data = request.get_json()
    user_message = data.get('message', '')

    if not user_message.strip():
        return jsonify({'reply': 'Please type something!'})

    answer, score = get_best_answer(user_message)
    return jsonify({'reply': answer, 'confidence': round(score, 2)})

# Set your ngrok authtoken here. Get it from https://dashboard.ngrok.com/get-started/your-authtoken
# You MUST replace the text "YOUR_AUTHTOKEN" below with your actual ngrok token.
# Example: ngrok.set_auth_token("2P7wXkY1Zc3bQ4sR5tU6vW7xY8z0A1B2C3D4E5F6")
ngrok.set_auth_token("3GOKuYqtl4EWJLneza9e8UmeI5R_6cJi1qcXM18FRekqktYex")

# Start ngrok tunnel to make this Colab server publicly accessible
public_url = ngrok.connect(5000)
print(f"🌍 Your chatbot is live at: {public_url}")

app.run(port=5000)

🌍 Your chatbot is live at: NgrokTunnel: "https://audacity-unfixed-bagel.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [12/Jul/2026 06:34:46] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [12/Jul/2026 06:34:55] "POST /chat HTTP/1.1" 200 -


@app.route('/') → this is your homepage — it shows the chat UI (from Cell 7)

@app.route('/chat') → this is the "waiter" — when the frontend sends a message here, it calls your get_best_answer() function and sends the reply back

ngrok.connect(5000) → since Colab itself can't show live websites to the outside world, ngrok creates a temporary public link (like https://abcd1234.ngrok.io) that anyone can open in a browser to use your chatbot

⚠️ Important: Run Cell 7 (below) first, then come back and run this Cell 6 — because this code needs the HTML_PAGE variable that Cell 7 creates.